In [3]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout, Input
from tensorflow.keras.callbacks import EarlyStopping

In [4]:
# 1. Cargar el dataset limpio de la Actividad 1
df = pd.read_csv('Dataset_limpio.csv')

# 2. Separar el dataset en variables predictoras (X) y variable objetivo (y = stroke)
X = df.drop(columns=['stroke'])
y = df['stroke']

# 3. Dividir los datos en conjuntos de entrenamiento y prueba (80/20) estratificados
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.20,
    random_state=42,
    stratify=y  # Mantiene idéntica la proporción de clases desbalanceadas
)

# 4. Escalado de variables continuas
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [5]:
# Construcción del modelo secuencial con capas ocultas y dropout
model = Sequential([
    Input(shape=(X_train_scaled.shape[1],)),
    Dense(64, activation='relu'),
    Dropout(0.3),  # Regularización para prevenir overfitting
    Dense(32, activation='relu'),
    Dropout(0.3),
    Dense(1, activation='sigmoid')  # Capa de salida para clasificación binaria [0, 1]
])

In [6]:
# 1. Compilación del modelo con métricas adecuadas para clases desbalanceadas
model.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=[
        tf.keras.metrics.AUC(name='auc'),
        tf.keras.metrics.Precision(name='precision'),
        tf.keras.metrics.Recall(name='recall'),
        tf.keras.metrics.BinaryAccuracy(name='accuracy')
    ]
)

# 2. Configurar EarlyStopping para detener el entrenamiento si no hay mejora
early_stopping = EarlyStopping(
    monitor='val_loss',
    patience=10,
    restore_best_weights=True
)

# 3. Entrenamiento del modelo
history = model.fit(
    X_train_scaled, y_train,
    epochs=50,
    batch_size=32,
    validation_split=0.2,
    callbacks=[early_stopping],
    verbose=1
)

Epoch 1/50
103/103 ━━━━━━━━━━━━━━━━━━━━ 5s 23ms/step - accuracy: 0.7853 - auc: 0.5038 - loss: 0.4559 - precision: 0.0565 - recall: 0.2303 - val_accuracy: 0.9425 - val_auc: 0.5846 - val_loss: 0.2178 - val_precision: 0.0000e+00 - val_recall: 0.0000e+00
Epoch 2/50
103/103 ━━━━━━━━━━━━━━━━━━━━ 2s 16ms/step - accuracy: 0.9535 - auc: 0.6391 - loss: 0.1936 - precision: 0.0000e+00 - recall: 0.0000e+00 - val_accuracy: 0.9425 - val_auc: 0.7183 - val_loss: 0.2029 - val_precision: 0.0000e+00 - val_recall: 0.0000e+00
Epoch 3/50
103/103 ━━━━━━━━━━━━━━━━━━━━ 2s 17ms/step - accuracy: 0.9529 - auc: 0.7074 - loss: 0.1837 - precision: 0.0000e+00 - recall: 0.0000e+00 - val_accuracy: 0.9425 - val_auc: 0.7563 - val_loss: 0.1990 - val_precision: 0.0000e+00 - val_recall: 0.0000e+00
Epoch 4/50
103/103 ━━━━━━━━━━━━━━━━━━━━ 2s 14ms/step - accuracy: 0.9532 - auc: 0.7521 - loss: 0.1746 - precision: 0.3333 - recall: 0.0066 - val_accuracy: 0.9425 - val_auc: 0.7692 - val_loss: 0.1999 - val_precision: 0.0000e+00 - val

In [7]:
# Evaluación final en el conjunto de prueba nunca antes visto
eval_results = model.evaluate(X_test_scaled, y_test)
for name, value in zip(model.metrics_names, eval_results):
    print(f"{name}: {value:.4f}")

32/32 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.9511 - auc: 0.8225 - loss: 0.1629 - precision: 0.0000e+00 - recall: 0.0000e+00
loss: 0.1629
compile_metrics: 0.8225
